# 🚔 Chicago Crime Analysis — Notebook Final

## Objectifs

1. **Prédire la sévérité** des crimes (GRAVE / MOYEN / MINEUR)
2. **Calculer le gain financier** de chaque bonne prédiction
3. **Tester le transfert** du modèle Chicago → NYC
4. **Prédire le crime parfait** — quel crime va probablement se produire demain à 23h dans le West Side ?

## Contexte Business
> *"La police de Chicago reçoit +200 000 appels par an. Notre système prédit la sévérité d'un crime en temps réel pour allouer les bonnes ressources au bon endroit — et estime les économies générées par chaque bonne prédiction."*

## Pipeline
```
Step 0 — K-Means   : Zone géographique (non supervisé)
Step 1 — RF        : Type de crime groupé (4 catégories)
Step 2 — XGBoost   : Sévérité (GRAVE / MOYEN / MINEUR)
Step 3 — RF Temps  : Prédiction temporelle (quel crime demain ?)
```


## ⚙️ 0 — Imports

In [ ]:
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib requests --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import requests, warnings, joblib
from io import StringIO
from datetime import datetime, timedelta

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi':120, 'figure.facecolor':'white'})
RANDOM_STATE = 42
print('✅ Imports OK')


## 📥 1 — Chargement des Données Chicago

In [ ]:
print('⏳ Chargement Chicago 2022–2026...')
r = requests.get(
    'https://data.cityofchicago.org/resource/ijzp-q8t2.csv',
    params={'$limit':200000,'$where':'year>=2022','$order':'date DESC'},
    timeout=120
)
df_raw = pd.read_csv(StringIO(r.text), low_memory=False)
print(f'✅ {len(df_raw):,} lignes chargées')
df_raw.head(3)


⏳ Chargement Chicago 2022–2026...


## 🧹 2 — Feature Engineering

In [ ]:
df = df_raw.copy()

# Temporel
df['date_parsed'] = pd.to_datetime(df['date'], errors='coerce')
df['hour']        = df['date_parsed'].dt.hour
df['day_of_week'] = df['date_parsed'].dt.dayofweek
df['month']       = df['date_parsed'].dt.month
df['year']        = df['date_parsed'].dt.year
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# Nettoyage
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df['domestic']  = df['domestic'].astype(bool).astype(int)
df = df.dropna(subset=['latitude','longitude','primary_type','hour'])
df = df[(df['latitude'] != 0) & (df['longitude'] != 0)]

# ── CIBLE : SÉVÉRITÉ (au lieu de l'arrestation) ──
def get_severity(crime):
    grave  = ['HOMICIDE','WEAPONS VIOLATION','ROBBERY','ASSAULT',
               'CRIMINAL SEXUAL ASSAULT','KIDNAPPING','ARSON']
    moyen  = ['BATTERY','BURGLARY','MOTOR VEHICLE THEFT',
               'NARCOTICS','STALKING','INTIMIDATION']
    if crime in grave: return 'GRAVE'
    if crime in moyen: return 'MOYEN'
    return 'MINEUR'

df['severity']      = df['primary_type'].apply(get_severity)
le_sev              = LabelEncoder()
df['sev_encoded']   = le_sev.fit_transform(df['severity'])

# ── REGROUPEMENT CRIMES EN 4 CATÉGORIES ──
def group_crime(crime):
    violent  = ['HOMICIDE','WEAPONS VIOLATION','ROBBERY','ASSAULT',
                'BATTERY','CRIMINAL SEXUAL ASSAULT','KIDNAPPING']
    property = ['THEFT','BURGLARY','MOTOR VEHICLE THEFT',
                'CRIMINAL DAMAGE','ARSON']
    drug     = ['NARCOTICS','OTHER NARCOTIC VIOLATION']
    if crime in violent:  return 'VIOLENT'
    if crime in property: return 'PROPERTY'
    if crime in drug:     return 'DRUG'
    return 'PUBLIC ORDER'

df['crime_group']   = df['primary_type'].apply(group_crime)
le_group            = LabelEncoder()
df['crime_encoded'] = le_group.fit_transform(df['crime_group'])

# Location group
def loc_group(loc):
    if pd.isna(loc): return 5
    loc = str(loc).upper()
    if any(w in loc for w in ['STREET','ALLEY','SIDEWALK']): return 0
    if any(w in loc for w in ['RESIDENCE','APARTMENT','HOUSE']): return 1
    if any(w in loc for w in ['STORE','RESTAURANT','BANK']): return 2
    if any(w in loc for w in ['CTA','BUS','TRAIN']): return 3
    if any(w in loc for w in ['PARK','SCHOOL','CHURCH']): return 4
    return 5
df['location_group'] = df['location_description'].apply(loc_group)

print(f'✅ {len(df):,} lignes après nettoyage')
print(f'\n🎯 Distribution sévérité :')
for s, n in df['severity'].value_counts().items():
    pct = n/len(df)*100
    bar = '█' * int(pct/3)
    print(f'  {s:<12} {n:>7,} ({pct:>5.1f}%) {bar}')
print(f'\n📂 Distribution groupes de crimes :')
print(df['crime_group'].value_counts())


In [ ]:
# K-Means
scaler_kmeans = StandardScaler()
X_geo = scaler_kmeans.fit_transform(df[['latitude','longitude']])
kmeans = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = kmeans.fit_predict(X_geo)

CLUSTER_LABELS = {0:'Zone A·Downtown',1:'Zone B·North',2:'Zone C·West',
                  3:'Zone D·South',4:'Zone E·Far South'}
print('✅ K-Means : 5 clusters')
print(df['cluster'].value_counts().sort_index())


## 📊 3 — EDA : Patterns de Sévérité

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
SEV_COLORS = {'GRAVE':'#ef4444','MOYEN':'#f59e0b','MINEUR':'#10b981'}

# Sévérité par heure
sev_hour = df.groupby(['hour','severity']).size().unstack(fill_value=0)
sev_hour_pct = sev_hour.div(sev_hour.sum(axis=1), axis=0)*100
for sev, col in SEV_COLORS.items():
    if sev in sev_hour_pct.columns:
        axes[0,0].plot(range(24), sev_hour_pct[sev], 'o-',
                      color=col, lw=2.5, ms=5, label=sev)
axes[0,0].set_title('Sévérité par Heure', fontweight='bold')
axes[0,0].set_xlabel('Heure'); axes[0,0].set_ylabel('% des crimes')
axes[0,0].legend(); axes[0,0].set_xticks(range(0,24,2))
axes[0,0].spines['top'].set_visible(False); axes[0,0].spines['right'].set_visible(False)

# Sévérité par zone K-Means
sev_cluster = df.groupby(['cluster','severity']).size().unstack(fill_value=0)
sev_cluster_pct = sev_cluster.div(sev_cluster.sum(axis=1), axis=0)*100
sev_cluster_pct.index = [CLUSTER_LABELS.get(i,f'Zone {i}') for i in sev_cluster_pct.index]
sev_cluster_pct.plot(kind='barh', ax=axes[0,1],
    color=[SEV_COLORS.get(c,'gray') for c in sev_cluster_pct.columns],
    edgecolor='white', width=0.7)
axes[0,1].set_title('Sévérité par Zone K-Means', fontweight='bold')
axes[0,1].set_xlabel('% des crimes')
axes[0,1].spines['top'].set_visible(False); axes[0,1].spines['right'].set_visible(False)

# Évolution temporelle
sev_year = df.groupby(['year','severity']).size().unstack(fill_value=0)
sev_year_pct = sev_year.div(sev_year.sum(axis=1),axis=0)*100
for sev, col in SEV_COLORS.items():
    if sev in sev_year_pct.columns:
        axes[1,0].plot(sev_year_pct.index, sev_year_pct[sev],
                      'o-', color=col, lw=2.5, ms=7, label=sev)
axes[1,0].set_title('Évolution Sévérité 2022→2026', fontweight='bold')
axes[1,0].set_xlabel('Année'); axes[1,0].set_ylabel('% des crimes')
axes[1,0].legend()
axes[1,0].spines['top'].set_visible(False); axes[1,0].spines['right'].set_visible(False)

# Sévérité par type de lieu
sev_loc = df.groupby(['location_group','severity']).size().unstack(fill_value=0)
loc_names = {0:'Rue',1:'Résidence',2:'Commerce',3:'Transport',4:'Public',5:'Autre'}
sev_loc.index = [loc_names.get(i,str(i)) for i in sev_loc.index]
sev_loc_pct = sev_loc.div(sev_loc.sum(axis=1),axis=0)*100
sev_loc_pct.plot(kind='bar', ax=axes[1,1],
    color=[SEV_COLORS.get(c,'gray') for c in sev_loc_pct.columns],
    edgecolor='white', width=0.7)
axes[1,1].set_title('Sévérité par Type de Lieu', fontweight='bold')
axes[1,1].set_xlabel('Type de lieu'); axes[1,1].set_ylabel('% des crimes')
axes[1,1].tick_params(axis='x', rotation=30)
axes[1,1].spines['top'].set_visible(False); axes[1,1].spines['right'].set_visible(False)

plt.suptitle('📊 Patterns de Sévérité — Chicago 2022–2026',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_final_01_severity_patterns.png', bbox_inches='tight')
plt.show()
print('✅ viz_final_01_severity_patterns.png')


## 🤖 4 — Modèles ML

In [ ]:
FEAT_S1 = ['latitude','longitude','hour','day_of_week','month',
           'is_weekend','cluster','location_group']
FEAT_S2 = FEAT_S1 + ['domestic','crime_encoded']

# Step 1 — Crime Group
df_s1 = df[FEAT_S1+['crime_encoded']].dropna()
X1_tr,X1_te,y1_tr,y1_te = train_test_split(
    df_s1[FEAT_S1], df_s1['crime_encoded'],
    test_size=.2, random_state=RANDOM_STATE, stratify=df_s1['crime_encoded'])

print('⏳ Step 1 — RF Crime Group...')
rf_crime = RandomForestClassifier(n_estimators=200, max_depth=12,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_crime.fit(X1_tr, y1_tr)
y1_pred   = rf_crime.predict(X1_te)
acc_s1    = accuracy_score(y1_te, y1_pred)
f1_s1     = f1_score(y1_te, y1_pred, average='weighted')
print(f'   ✅ Accuracy: {acc_s1*100:.1f}%  F1: {f1_s1*100:.1f}%')

# Prédire sur tout le dataset
df['crime_predicted'] = rf_crime.predict(df[FEAT_S1].fillna(0))

# Step 2 — Sévérité
df_s2 = df[FEAT_S2+['sev_encoded']].dropna()
X2_tr,X2_te,y2_tr,y2_te = train_test_split(
    df_s2[FEAT_S2], df_s2['sev_encoded'],
    test_size=.2, random_state=RANDOM_STATE, stratify=df_s2['sev_encoded'])

ratio = (y2_tr==0).sum()/(y2_tr==1).sum()
print('⏳ Step 2 — XGBoost Sévérité...')
xgb_sev = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, eval_metric='mlogloss', verbosity=0, n_jobs=-1)
xgb_sev.fit(X2_tr, y2_tr, eval_set=[(X2_te, y2_te)], verbose=False)
y2_pred   = xgb_sev.predict(X2_te)
acc_s2    = accuracy_score(y2_te, y2_pred)
f1_s2     = f1_score(y2_te, y2_pred, average='weighted')
print(f'   ✅ Accuracy: {acc_s2*100:.1f}%  F1: {f1_s2*100:.1f}%')

print(f'\n📊 RÉSULTATS FINAUX :')
print(f'   Step 1 RF Crime Group : {acc_s1*100:.1f}% accuracy')
print(f'   Step 2 XGB Sévérité   : {acc_s2*100:.1f}% accuracy')
print(f'\n{classification_report(y2_te, y2_pred, target_names=le_sev.classes_)}')


In [ ]:
# ── VIZ : Matrice de confusion + Feature importances ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y2_te, y2_pred, normalize='true')
sns.heatmap(cm, annot=True, fmt='.2f', cmap='RdYlGn',
            ax=axes[0], xticklabels=le_sev.classes_,
            yticklabels=le_sev.classes_, linewidths=0.5)
axes[0].set_title('Matrice de Confusion — Sévérité', fontweight='bold')
axes[0].set_xlabel('Prédit'); axes[0].set_ylabel('Réel')

fi = pd.Series(xgb_sev.feature_importances_, index=FEAT_S2).sort_values()
col_fi = ['#ef4444' if v > fi.median() else '#e2e8f0' for v in fi.values]
axes[1].barh(fi.index, fi.values*100, color=col_fi, edgecolor='white')
axes[1].set_xlabel('Importance (%)'); axes[1].set_title('Feature Importances — XGBoost', fontweight='bold')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('🤖 Évaluation XGBoost — Prédiction Sévérité',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_final_02_model_eval.png', bbox_inches='tight')
plt.show()


## 💰 5 — Analyse Coût/Gain par Prédiction

### Hypothèses basées sur données publiques Chicago PD
| Intervention | Coût estimé |
|---|---|
| Unité spécialisée (GRAVE) | $800 |
| Patrouille standard (MOYEN) | $250 |
| Traitement administratif (MINEUR) | $50 |
| Coût d'une mauvaise classification | $400 gaspillés |


In [ ]:
# Coûts réels par type d'intervention (données publiques Chicago PD budget)
COST = {
    'GRAVE':  800,   # unité spécialisée
    'MOYEN':  250,   # patrouille standard
    'MINEUR':  50,   # traitement administratif
}
WASTE_COST = 400    # coût d'une mauvaise classification

# Classes réelles et prédites
true_labels = le_sev.inverse_transform(y2_te)
pred_labels = le_sev.inverse_transform(y2_pred)

results = []
for true, pred in zip(true_labels, pred_labels):
    correct     = true == pred
    cost_saved  = COST[true] - COST[pred] if not correct else COST[true]
    waste       = WASTE_COST if not correct else 0
    net_gain    = cost_saved - waste if not correct else COST[true]
    results.append({'true':true,'pred':pred,'correct':correct,
                    'cost_optimal':COST[true],'waste':waste,'net':net_gain})

df_costs = pd.DataFrame(results)

total_crimes   = len(df_costs)
correct_preds  = df_costs['correct'].sum()
wrong_preds    = total_crimes - correct_preds
total_savings  = df_costs[df_costs['correct']]['cost_optimal'].sum()
total_waste    = df_costs[~df_costs['correct']]['waste'].sum()
net_total      = total_savings - total_waste
baseline_cost  = sum(COST[c] for c in true_labels)  # sans modèle → tout traité en MOYEN

# Scaling annuel
scale = 200000 / total_crimes
annual_net  = net_total * scale
annual_base = baseline_cost * scale

print('='*60)
print('  ANALYSE COÛT/GAIN')
print('='*60)
print(f'\n  Sur {total_crimes:,} crimes analysés :')
print(f'  ✅ Bonnes prédictions : {correct_preds:,} ({correct_preds/total_crimes*100:.1f}%)')
print(f'  ❌ Mauvaises prédictions : {wrong_preds:,} ({wrong_preds/total_crimes*100:.1f}%)')
print(f'\n  Économies sur bonnes prédictions : ${total_savings:,.0f}')
print(f'  Coût des mauvaises prédictions   : ${total_waste:,.0f}')
print(f'  GAIN NET (test set)              : ${net_total:,.0f}')
print(f'\n  📈 PROJECTION ANNUELLE (200K crimes) :')
print(f'  Sans modèle (traitement uniforme) : ${annual_base:,.0f}')
print(f'  Avec notre modèle                 : ${annual_base - annual_net:,.0f}')
print(f'  💰 ÉCONOMIES ANNUELLES ESTIMÉES  : ${annual_net:,.0f}')
print(f'\n  → Soit ${annual_net/200000:.0f} d\'économie par crime traité')


In [ ]:
# ── VIZ : Coût/gain par catégorie ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gain net par catégorie de sévérité réelle
gain_by_sev = df_costs.groupby('true')['net'].sum()
colors_gain = ['#10b981' if v > 0 else '#ef4444' for v in gain_by_sev.values]
bars = axes[0].bar(gain_by_sev.index, gain_by_sev.values/1000,
                   color=colors_gain, edgecolor='white', width=0.5)
axes[0].set_ylabel('Gain net ($K)'); axes[0].set_xlabel('Sévérité réelle')
axes[0].set_title('Gain Net par Catégorie', fontweight='bold')
for bar, val in zip(bars, gain_by_sev.values/1000):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height() + (1 if val >= 0 else -3),
                 f'${val:.0f}K', ha='center', fontweight='bold', fontsize=10)
axes[0].axhline(y=0, color='black', lw=0.8)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Confusion matrix des coûts
cost_matrix = pd.crosstab(df_costs['true'], df_costs['pred'],
                          values=df_costs['waste'], aggfunc='sum').fillna(0)/1000
sns.heatmap(cost_matrix, annot=True, fmt='.0f', cmap='Reds',
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Coût des Erreurs ($K) par Catégorie', fontweight='bold')
axes[1].set_xlabel('Prédit'); axes[1].set_ylabel('Réel')

plt.suptitle('💰 Analyse Coût/Gain — Impact Financier du Modèle',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_final_03_cost_analysis.png', bbox_inches='tight')
plt.show()
print('✅ viz_final_03_cost_analysis.png')


## 🌆 6 — Transfer Learning Chicago → NYC

### Principe
On applique le modèle entraîné sur Chicago **directement sur NYC** sans réentraîner.
Puis on compare avec un modèle réentraîné sur NYC.
→ Mesure le **data drift** entre les deux villes.


In [ ]:
print('⏳ Chargement données NYC...')
try:
    r_nyc = requests.get(
        'https://data.cityofnewyork.us/resource/uip8-fykc.json',
        params={'$limit':100000,'$order':'arrest_date DESC',
                '$select':'arrest_date,ofns_desc,law_cat_cd,arrest_boro,latitude,longitude'},
        timeout=120
    )
    nyc = pd.DataFrame(r_nyc.json())
    print(f'✅ NYC : {len(nyc):,} lignes')
except Exception as e:
    print(f'⚠️ {e}')

# Feature engineering NYC (même pipeline)
nyc['latitude']  = pd.to_numeric(nyc['latitude'],  errors='coerce')
nyc['longitude'] = pd.to_numeric(nyc['longitude'], errors='coerce')
nyc = nyc.dropna(subset=['latitude','longitude','ofns_desc','law_cat_cd'])
nyc = nyc[(nyc['latitude'] != 0) & (nyc['longitude'] != 0)]

nyc['date_parsed'] = pd.to_datetime(nyc['arrest_date'], errors='coerce')
nyc['hour']        = nyc['date_parsed'].dt.hour.fillna(12).astype(int)
nyc['day_of_week'] = nyc['date_parsed'].dt.dayofweek.fillna(0).astype(int)
nyc['month']       = nyc['date_parsed'].dt.month.fillna(6).astype(int)
nyc['is_weekend']  = (nyc['day_of_week'] >= 5).astype(int)
nyc['domestic']    = 0  # pas disponible dans NYC

# Mapping boroughs → location_group (même encodage que Chicago)
boro_map = {'M':0,'K':1,'B':0,'Q':3,'S':4}
nyc['location_group'] = nyc['arrest_boro'].map(boro_map).fillna(0).astype(int)

# Sévérité NYC (Felony=GRAVE, Misdemeanor=MOYEN, Violation=MINEUR)
sev_nyc_map = {'F':'GRAVE','M':'MOYEN','V':'MINEUR','I':'MINEUR'}
nyc['severity']     = nyc['law_cat_cd'].map(sev_nyc_map).fillna('MINEUR')
nyc['sev_encoded']  = le_sev.transform(nyc['severity'].apply(
    lambda x: x if x in le_sev.classes_ else 'MINEUR'))

# K-Means NYC (nouveau scaler car coordonnées différentes)
scaler_nyc = StandardScaler()
nyc['cluster'] = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=10).fit_predict(
    scaler_nyc.fit_transform(nyc[['latitude','longitude']]))

# Crime group NYC
def group_nyc(crime):
    if pd.isna(crime): return 'PUBLIC ORDER'
    crime = str(crime).upper()
    if any(w in crime for w in ['ASSAULT','ROBBERY','WEAPON','HOMICIDE','RAPE']): return 'VIOLENT'
    if any(w in crime for w in ['THEFT','BURGLARY','LARCENY','VEHICLE']): return 'PROPERTY'
    if any(w in crime for w in ['DRUG','NARCOTIC','CONTROLLED']): return 'DRUG'
    return 'PUBLIC ORDER'

nyc['crime_group']   = nyc['ofns_desc'].apply(group_nyc)
# Encoder avec le même LabelEncoder que Chicago
nyc['crime_encoded'] = le_group.transform(nyc['crime_group'].apply(
    lambda x: x if x in le_group.classes_ else le_group.classes_[0]))

print(f'✅ NYC préparé : {len(nyc):,} lignes')
print(nyc['severity'].value_counts())


In [ ]:
# ── Transfer : modèle Chicago sur NYC ──
nyc_valid = nyc[FEAT_S2+['sev_encoded']].dropna()
X_nyc = nyc_valid[FEAT_S2]; y_nyc = nyc_valid['sev_encoded']

# Prédiction directe (sans réentraînement)
y_nyc_pred_transfer = xgb_sev.predict(X_nyc)
acc_transfer = accuracy_score(y_nyc, y_nyc_pred_transfer)
f1_transfer  = f1_score(y_nyc, y_nyc_pred_transfer, average='weighted')

# Modèle réentraîné sur NYC
X_nyc_tr, X_nyc_te, y_nyc_tr, y_nyc_te = train_test_split(
    X_nyc, y_nyc, test_size=.2, random_state=RANDOM_STATE, stratify=y_nyc)
xgb_nyc = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
    random_state=RANDOM_STATE, eval_metric='mlogloss', verbosity=0, n_jobs=-1)
xgb_nyc.fit(X_nyc_tr, y_nyc_tr)
y_nyc_retrained = xgb_nyc.predict(X_nyc_te)
acc_retrained = accuracy_score(y_nyc_te, y_nyc_retrained)
f1_retrained  = f1_score(y_nyc_te, y_nyc_retrained, average='weighted')

print('='*55)
print('  TRANSFER LEARNING CHICAGO → NYC')
print('='*55)
print(f'  Modèle Chicago (sans réentraîner) : {acc_transfer*100:.1f}% accuracy')
print(f'  Modèle réentraîné sur NYC         : {acc_retrained*100:.1f}% accuracy')
print(f'  Data Drift (écart)                : {(acc_retrained-acc_transfer)*100:.1f}%')
print()
if (acc_retrained - acc_transfer) < 5:
    print('  ✅ Data drift faible — le modèle Chicago généralise bien sur NYC !')
elif (acc_retrained - acc_transfer) < 10:
    print('  ⚠️  Data drift modéré — réentraînement recommandé pour NYC')
else:
    print('  🔴 Data drift fort — les deux villes sont trop différentes')


In [ ]:
# ── VIZ : Comparaison Transfer ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy comparison
models   = ['Chicago\n(train)', 'NYC Transfer\n(sans réentraîn.)', 'NYC Natif\n(réentraîné)']
accs     = [acc_s2*100, acc_transfer*100, acc_retrained*100]
colors   = ['#3b82f6', '#f59e0b', '#10b981']
bars = axes[0].bar(models, accs, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_ylim(0, 100); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Transfer Learning — Accuracy', fontweight='bold')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# Distribution sévérité Chicago vs NYC
chi_sev = df['severity'].value_counts(normalize=True)*100
nyc_sev = nyc['severity'].value_counts(normalize=True)*100
x = np.arange(len(le_sev.classes_))
w = 0.35
axes[1].bar(x-w/2, [chi_sev.get(s,0) for s in le_sev.classes_], w,
           color='#3b82f6', label='Chicago', edgecolor='white', alpha=0.85)
axes[1].bar(x+w/2, [nyc_sev.get(s,0) for s in le_sev.classes_], w,
           color='#ef4444', label='NYC', edgecolor='white', alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(le_sev.classes_)
axes[1].set_ylabel('% des crimes'); axes[1].set_title('Distribution Sévérité : Chicago vs NYC', fontweight='bold')
axes[1].legend(); axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('🌆 Transfer Learning — Chicago → NYC', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_final_04_transfer.png', bbox_inches='tight')
plt.show()
print('✅ viz_final_04_transfer.png')


## 🔮 7 — Prédiction Temporelle : Quel Crime Demain ?

> *"Basé sur les patterns historiques, quel crime va probablement se produire demain à 23h dans le West Side ?"*

### Principe
On entraîne un modèle qui prédit le **type de crime** à partir de :
- L'heure, le jour, le mois
- La zone géographique (cluster K-Means)
- Le type de lieu

C'est l'inverse du pipeline habituel — on donne le contexte temporel/spatial et on prédit le crime.


In [ ]:
# ── Modèle de prédiction temporelle ──
FEAT_TEMPORAL = ['hour','day_of_week','month','is_weekend','cluster','location_group']

df_temp = df[FEAT_TEMPORAL+['crime_encoded','sev_encoded']].dropna()
X_t = df_temp[FEAT_TEMPORAL]; y_t = df_temp['crime_encoded']

X_t_tr, X_t_te, y_t_tr, y_t_te = train_test_split(
    X_t, y_t, test_size=.2, random_state=RANDOM_STATE, stratify=y_t)

print('⏳ Entraînement modèle prédiction temporelle...')
rf_temporal = RandomForestClassifier(n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_temporal.fit(X_t_tr, y_t_tr)

acc_temp = accuracy_score(y_t_te, rf_temporal.predict(X_t_te))
f1_temp  = f1_score(y_t_te, rf_temporal.predict(X_t_te), average='weighted')
print(f'✅ Accuracy: {acc_temp*100:.1f}%  F1: {f1_temp*100:.1f}%')

# ── Fonction de prédiction ──
def predict_crime_tomorrow(hour, day_name, month, zone_name, location_name, top_n=3):
    day_map  = {'Lundi':0,'Mardi':1,'Mercredi':2,'Jeudi':3,'Vendredi':4,'Samedi':5,'Dimanche':6}
    zone_map = {'Downtown':0,'North Side':1,'West Side':2,'South Side':3,'Far South':4}
    loc_map  = {'Rue':0,'Résidence':1,'Commerce':2,'Transport':3,'Public/Parc':4,'Autre':5}

    day_num      = day_map.get(day_name, 0)
    cluster_num  = zone_map.get(zone_name, 0)
    loc_num      = loc_map.get(location_name, 0)
    is_we        = int(day_num >= 5)

    x = pd.DataFrame([{'hour':hour,'day_of_week':day_num,'month':month,
                        'is_weekend':is_we,'cluster':cluster_num,
                        'location_group':loc_num}])[FEAT_TEMPORAL]

    probas = rf_temporal.predict_proba(x)[0]
    top_idx = probas.argsort()[::-1][:top_n]
    top_crimes = [(le_group.inverse_transform([i])[0], round(float(probas[i])*100,1))
                  for i in top_idx]

    # Prédire aussi la sévérité du crime le plus probable
    crime_pred = top_idx[0]
    x_sev = pd.DataFrame([{'hour':hour,'day_of_week':day_num,'month':month,
                            'is_weekend':is_we,'cluster':cluster_num,
                            'location_group':loc_num,'domestic':0,
                            'crime_encoded':crime_pred}])[FEAT_S2]
    sev_pred  = xgb_sev.predict(x_sev)[0]
    sev_label = le_sev.inverse_transform([sev_pred])[0]
    sev_proba = xgb_sev.predict_proba(x_sev)[0].max()

    return {
        'contexte':  f'{day_name} {hour}h00 — {zone_name} — {location_name}',
        'top_crimes': top_crimes,
        'sévérité':   sev_label,
        'conf_sév':   f'{sev_proba*100:.1f}%',
        'intervention': {
            'GRAVE': '🚨 Unité spécialisée immédiate',
            'MOYEN': '🚔 Patrouille standard',
            'MINEUR': '📋 Traitement administratif'
        }.get(sev_label,'')
    }

print('✅ Fonction predict_crime_tomorrow() prête')


In [ ]:
# ── SCÉNARIOS DE PRÉDICTION ──
tomorrow = datetime.now() + timedelta(days=1)
day_names = ['Lundi','Mardi','Mercredi','Jeudi','Vendredi','Samedi','Dimanche']
tomorrow_name = day_names[tomorrow.weekday()]
tomorrow_month = tomorrow.month

print('='*65)
print('  🔮 PRÉDICTIONS TEMPORELLES — DEMAIN')
print('='*65)

scenarios = [
    (23, tomorrow_name, tomorrow_month, 'West Side',   'Rue',        'Scénario 1 — West Side, Nuit'),
    (14, tomorrow_name, tomorrow_month, 'Downtown',    'Commerce',   'Scénario 2 — Downtown, Après-midi'),
    (8,  tomorrow_name, tomorrow_month, 'North Side',  'Transport',  'Scénario 3 — North Side, Matin'),
    (22, 'Samedi',      tomorrow_month, 'South Side',  'Rue',        'Scénario 4 — South Side, Nuit weekend'),
    (3,  'Dimanche',    tomorrow_month, 'Far South',   'Résidence',  'Scénario 5 — Far South, 3h du matin'),
]

for hour, day, month, zone, loc, title in scenarios:
    result = predict_crime_tomorrow(hour, day, month, zone, loc)
    print(f'\n📍 {title}')
    print(f'   Contexte    : {result["contexte"]}')
    print(f'   Crimes probables :')
    for crime, proba in result['top_crimes']:
        bar = '█' * int(proba/5)
        print(f'     {crime:<15} {proba:>5.1f}%  {bar}')
    print(f'   Sévérité    : {result["sévérité"]} ({result["conf_sév"]})')
    print(f'   Action      : {result["intervention"]}')


In [ ]:
# ── VIZ : Heatmap criminalité par heure et zone ──
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Heatmap heure × zone
pivot = df.groupby(['cluster','hour']).size().unstack(fill_value=0)
pivot.index = [CLUSTER_LABELS.get(i,f'Zone {i}') for i in pivot.index]
sns.heatmap(pivot, cmap='YlOrRd', ax=axes[0], linewidths=0.3,
            cbar_kws={'label':'Nombre de crimes'})
axes[0].set_title('Heatmap Criminalité — Zone × Heure', fontweight='bold')
axes[0].set_xlabel('Heure de la journée'); axes[0].set_ylabel('Zone K-Means')

# Heatmap sévérité × heure
sev_pivot = df.groupby(['hour','severity']).size().unstack(fill_value=0)
sev_pivot_pct = sev_pivot.div(sev_pivot.sum(axis=1),axis=0)*100
sev_pivot_pct[['GRAVE','MOYEN','MINEUR']].plot(
    kind='area', ax=axes[1], alpha=0.6,
    color=['#ef4444','#f59e0b','#10b981'])
axes[1].set_title('Sévérité par Heure (% cumulé)', fontweight='bold')
axes[1].set_xlabel('Heure'); axes[1].set_ylabel('% des crimes')
axes[1].legend(loc='upper left'); axes[1].set_xticks(range(0,24,2))
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('🗓️ Patterns Temporels et Géographiques',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_final_05_temporal_heatmap.png', bbox_inches='tight')
plt.show()
print('✅ viz_final_05_temporal_heatmap.png')


## 💡 8 — Insights Finaux

In [ ]:
print('='*65)
print('  INSIGHTS FINAUX')
print('='*65)

# Insight 1 : Sévérité
sev_dist = df['severity'].value_counts(normalize=True)*100
print(f'\n⚠️  INSIGHT 1 — Distribution de la sévérité')
for s, p in sev_dist.items():
    print(f'   {s:<10} : {p:.1f}%')
grave_zone = df.groupby('cluster')['sev_encoded'].apply(
    lambda x: (le_sev.inverse_transform(x)=='GRAVE').mean()*100)
print(f'\n   Zone avec le plus de crimes GRAVES : Cluster {grave_zone.idxmax()}')
print(f'   ({grave_zone.max():.1f}% des crimes sont GRAVES dans cette zone)')

# Insight 2 : Temporel
night_grave = df[df['hour'].between(22,5) if False else
               df['hour'].isin(list(range(22,24))+list(range(0,6)))]['severity'].value_counts(normalize=True)*100
day_grave   = df[df['hour'].between(9,17)]['severity'].value_counts(normalize=True)*100
print(f'\n⏰ INSIGHT 2 — Impact de l\'heure sur la sévérité')
print(f'   Nuit (22h-6h) crimes GRAVES  : {night_grave.get("GRAVE",0):.1f}%')
print(f'   Jour (9h-17h) crimes GRAVES  : {day_grave.get("GRAVE",0):.1f}%')

# Insight 3 : Financier
print(f'\n💰 INSIGHT 3 — Impact financier')
print(f'   Step 2 XGBoost accuracy      : {acc_s2*100:.1f}%')
print(f'   Économies annuelles estimées : ${annual_net:,.0f}')
print(f'   Gain par crime traité        : ${annual_net/200000:.0f}')

# Insight 4 : Transfer
print(f'\n🌆 INSIGHT 4 — Généralisation inter-villes')
print(f'   Chicago → NYC transfer       : {acc_transfer*100:.1f}% accuracy')
print(f'   NYC réentraîné               : {acc_retrained*100:.1f}% accuracy')
drift = (acc_retrained-acc_transfer)*100
print(f'   Data drift                   : {drift:.1f}% — {"faible ✅" if drift<5 else "modéré ⚠️"}')

print(f'\n✅ CONCLUSION :')
print(f'   Notre pipeline K-Means → RF → XGBoost prédit la sévérité')
print(f'   des crimes avec {acc_s2*100:.1f}% d\'accuracy, génère ~${annual_net:,.0f}/an')
print(f'   d\'économies, et se transfère sur NYC avec {drift:.1f}% de drift.')


## 💾 9 — Sauvegarde

In [ ]:
joblib.dump(rf_crime,    'rf_crime_group.pkl')
joblib.dump(xgb_sev,     'xgb_severity.pkl')
joblib.dump(rf_temporal, 'rf_temporal.pkl')
joblib.dump(kmeans,      'kmeans_chicago.pkl')
joblib.dump(scaler_kmeans,'scaler_kmeans.pkl')
joblib.dump(le_group,    'le_crime_group.pkl')
joblib.dump(le_sev,      'le_severity.pkl')

print('✅ Modèles sauvegardés :')
print('   rf_crime_group.pkl  — Step 1 : groupe de crime')
print('   xgb_severity.pkl    — Step 2 : sévérité')
print('   rf_temporal.pkl     — Step 3 : prédiction temporelle')
print()
import os
print('📊 Visualisations :')
for f in sorted(os.listdir('.')):
    if f.startswith('viz_final_') and f.endswith('.png'):
        print(f'   {f}')
